In [1]:
# Cell 0: Master Protocol Generation and Data Balancing Script
import os
import pandas as pd
import random

# ==============================================================================
# 1. Path Configuration (Verify these absolute paths exist on your local machine)
# ==============================================================================
BASE_WF = "/home/jesse/Projects/MyProjects/2026_FinQuest/Finquest_Wav2Vec_Deepfake/WaveFake"
BASE_ASV = "/home/jesse/Projects/MyProjects/2026_FinQuest/Finquest_Wav2Vec_Deepfake/LA"
BASE_WILD = "/home/jesse/Projects/MyProjects/2026_FinQuest/Finquest_Wav2Vec_Deepfake/in_the_wild"
OUTPUT_DIR = "./protocol"

# Ensure output directory exists for version control
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)
    print(f"📁 Created directory: {OUTPUT_DIR}")

def build_protocol():
    """
    Consolidates multiple datasets (WaveFake, ASVspoof, In-the-wild) into a 
    unified CSV protocol with balanced Real/Fake labels for training.
    """
    master_list = []

    # Utility function: Retrieve file paths with specific audio extensions
    def get_files(path, ext=(".wav", ".WAV", ".flac")):
        if not os.path.exists(path):
            print(f"⚠️  Warning: Directory not found: {path}")
            return []
        return [os.path.join(path, f) for f in os.listdir(path) if f.lower().endswith(ext)]

    # ---- 1. LJSpeech Real (Total ~13,100 samples) ----
    ljs_real_path = os.path.join(BASE_WF, "LJSpeech-1.1_real")
    ljs_real_files = get_files(ljs_real_path)
    print(f"-> LJ Real Samples found: {len(ljs_real_files)}")
    for f in ljs_real_files:
        master_list.append({"path": f, "label": 1, "model_name": "original", "source": "LJ_Real"})

    # ---- 2. WaveFake LJ Fake (Stratified Sampling) ----
    # We sample 1,872 unique utterances per synthesis model to maintain diversity.
    fake_folders = [
        "ljspeech_full_band_melgan", "ljspeech_hifiGAN", "ljspeech_melgan",
        "ljspeech_melgan_large", "ljspeech_multi_band_melgan",
        "ljspeech_parallel_wavegan", "ljspeech_waveglow"
    ]
    for i, folder in enumerate(fake_folders):
        f_path = os.path.join(BASE_WF, folder)
        f_files = sorted(get_files(f_path))
        # Non-overlapping sampling logic
        sample = f_files[i*1872 : (i+1)*1872]
        model_tag = folder.split('_')[-1]
        for f in sample:
            master_list.append({"path": f, "label": 0, "model_name": model_tag, "source": "WF_LJ"})
    print(f"-> Total after WaveFake LJ integration: {len(master_list)}")

    # ---- 3. JSUT Dataset (1:2 Ratio Integration) ----
    jsut_real_path = os.path.join(BASE_WF, "jsut_real") 
    jsut_real_files = get_files(jsut_real_path)
    print(f"-> JSUT Real Samples found: {len(jsut_real_files)}")
    for f in jsut_real_files:
        master_list.append({"path": f, "label": 1, "model_name": "original", "source": "JSUT_Real"})

    # Adding Japanese synthetic speech for cross-lingual robustness
    for folder in ["jsut_multi_band_melgan", "jsut_parallel_wavegan"]:
        f_files = get_files(os.path.join(BASE_WF, folder))
        for f in f_files:
            master_list.append({"path": f, "label": 0, "model_name": folder.split('_')[-1], "source": "JSUT_Fake"})

    # ---- 4. Common Voice Prompts (Stochastic Sampling of 2,000) ----
    cv_path = os.path.join(BASE_WF, "common_voices_prompts_from_conformer_fastspeech2_pwg_ljspeech")
    cv_files = get_files(cv_path)
    if cv_files:
        sample_cv = random.sample(cv_files, min(2000, len(cv_files)))
        for f in sample_cv:
            master_list.append({"path": f, "label": 0, "model_name": "fastspeech2_pwg", "source": "WF_CV"})

    # ---- 5. In-the-wild Data (External Generalization Set) ----
    wild_csv = os.path.join(BASE_WILD, "meta.csv")
    if os.path.exists(wild_csv):
        # Auto-detecting delimiter for meta.csv (comma or tab)
        df_wild = pd.read_csv(wild_csv, sep=None, engine='python')
        for _, row in df_wild.iterrows():
            fname = str(row['file'])
            f_path = os.path.join(BASE_WILD, fname)
            master_list.append({
                "path": f_path,
                "label": 1 if row['label'] == 'bona-fide' else 0,
                "model_name": "unknown_wild",
                "source": "Wild"
            })

    # ---- 6. ASVspoof 2019 (Balanced Sampling) ----
    # To prevent class imbalance, we perform 1:1 sampling of Real vs Spoof samples.
    asv_prot = os.path.join(BASE_ASV, "ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt")
    if os.path.exists(asv_prot):
        asv_data = []
        with open(asv_prot, "r") as f:
            for line in f:
                parts = line.strip().split()
                # Mapping protocol IDs to physical flac file paths
                path = os.path.join(BASE_ASV, "ASVspoof2019_LA_train/flac", parts[1] + ".flac")
                asv_data.append({
                    "path": path, 
                    "label": 1 if parts[4] == "bonafide" else 0, 
                    "model_name": parts[3] if parts[4] == "spoof" else "original", 
                    "source": "ASV"
                })
        
        df_asv = pd.DataFrame(asv_data)
        reals = df_asv[df_asv['label'] == 1]
        # Under-sampling fake data to match real samples count for parity
        fakes = df_asv[df_asv['label'] == 0].sample(n=len(reals), random_state=42) 
        master_list.extend(reals.to_dict('records'))
        master_list.extend(fakes.to_dict('records'))

    # ---- 7. Export Master Protocol ----
    df_final = pd.DataFrame(master_list)
    final_path = os.path.join(OUTPUT_DIR, "master_protocol.csv")
    df_final.to_csv(final_path, index=False)

    print("\n" + "="*50)
    print("📋 FINAL DATASET GENERATION REPORT")
    print("="*50)
    print(f"Total Samples    : {len(df_final)}")
    print(f"Real (Label 1)   : {len(df_final[df_final['label']==1])}")
    print(f"Fake (Label 0)   : {len(df_final[df_final['label']==0])}")
    print("-" * 50)
    print("Distribution by Source:")
    print(df_final['source'].value_counts())
    print("="*50)
    print(f"✅ Success: Master protocol saved to {final_path}")

if __name__ == "__main__":
    build_protocol()

📁 Created directory: ./protocol
-> LJ Real Samples found: 13100
-> Total after WaveFake LJ integration: 26200
-> JSUT Real Samples found: 5000

📋 FINAL DATASET GENERATION REPORT
Total Samples    : 48360
Real (Label 1)   : 20680
Fake (Label 0)   : 27680
--------------------------------------------------
Distribution by Source:
source
LJ_Real      13100
WF_LJ        13100
JSUT_Fake    10000
ASV           5160
JSUT_Real     5000
WF_CV         2000
Name: count, dtype: int64
✅ Success: Master protocol saved to ./protocol/master_protocol.csv
